# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring**

My lane (from ML-02) is Refresh / Content Opportunity Scoring, and the decision was "which pages should the editor review first this week, given a fixed review budget." That is a "which ones first" question, which maps to ranking/scoring, not plain classification.

Underneath the ranking, I still need a binary signal to rank *by*: whether a page is currently trending down (`is_declining_label`). So the pipeline is a classification model feeding a ranking output; the deliverable an editor actually uses is the ranked queue, not the raw yes/no label.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), \
    "starter CSV not found -- are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("rows:", len(df))
print("task framing check:")
print("- underlying label: is_declining_label (binary, from trend_direction)")
print("- deliverable: a ranked queue of content_id ordered by refresh priority")
print("- this is Ranking/Scoring built on top of a Classification target")

rows: 30000
task framing check:
- underlying label: is_declining_label (binary, from trend_direction)
- deliverable: a ranked queue of content_id ordered by refresh priority
- this is Ranking/Scoring built on top of a Classification target


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`**, defined as `trend_direction == 'down'`.

This is an **observed outcome**, not a hand-defined rule: `trend_direction` comes from comparing each page's trailing 30-day performance against its prior 30-day window (real clicks/impressions/sessions movement), not from someone deciding by eye what "needs a refresh" looks like.

One honest caveat: the down/flat/up split itself is a threshold choice on a continuous change (`trend_pct`), so it inherits whatever cutoff the data pipeline used. That's a much softer assumption than a hand-written business rule, but it's not zero-assumption either, worth naming rather than hiding.

In [2]:
print(df['trend_direction'].value_counts())
print("\ndeclining label rate:", round(df['is_declining_label'].mean(), 3))

# confirm this is genuinely an observed trailing-window comparison, not a static field
window_cols = ['impressions_last_30d','impressions_prev_30d','clicks_last_30d','clicks_prev_30d',
               'sessions_last_30d','sessions_prev_30d','trend_pct']
print("\nsupporting trailing-window columns exist:", all(c in df.columns for c in window_cols))
print(df[window_cols + ['trend_direction']].head(3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

declining label rate: 0.542

supporting trailing-window columns exist: True
   impressions_last_30d  impressions_prev_30d  clicks_last_30d  \
0                   578                   987                2   
1                  2501                  5915                2   
2                  2382                  6089                1   

   clicks_prev_30d  sessions_last_30d  sessions_prev_30d  trend_pct  \
0               13                  2                  9      -41.4   
1                1                  3                  2      -57.7   
2                3                  1                  3      -60.9   

  trend_direction  
0            down  
1            down  
2            down  


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

From ML-02, the editor can only deep-review a limited number of pages per week (I used 50 as the realistic weekly capacity). That means what matters isn't overall accuracy across all 30,000 pages, it's whether the top 50 the editor actually opens are genuinely declining. ROC-AUC or recall over the whole catalog would reward a model that's good "on average" while still wasting the editor's actual weekly slots.

This also matches how the reference pipeline in this repo evaluates its own models (`outputs/model_report.md` picks its best model by `precision_at_50`), so I'm not inventing a new success bar, I'm reusing the one this problem already earned.

In [4]:
weekly_review_capacity = 50
top_k = df.sort_values('impressions_90d', ascending=False).head(weekly_review_capacity)
naive_precision_at_50 = top_k['is_declining_label'].mean()
print(f"'good' means: precision@{weekly_review_capacity} clearly above the {df['is_declining_label'].mean():.1%} base rate")
print(f"(just sorting by impressions alone gets {naive_precision_at_50:.1%} precision@50, for comparison)")

'good' means: precision@50 clearly above the 54.2% base rate
(just sorting by impressions alone gets 42.0% precision@50, for comparison)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one published content item (`content_id`) belonging to one client (`client_id`).**

Not a keyword, not a session, not a client. Each row is a single live page whose refresh-worthiness the editor is deciding on.

In [5]:
print("shape:", df.shape)
print("unique content_id:", df['content_id'].nunique(), "| unique client_id:", df['client_id'].nunique())
df[['content_id','client_id','content_type','main_intent','avg_position','trend_direction','is_declining_label']].head(5)

shape: (30000, 45)
unique content_id: 30000 | unique client_id: 32


,content_id,client_id,content_type,main_intent,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,44.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A plausible hand-written rule does not beat guessing.**

I tried the rule a human would actually write: flag pages that are *both* poorly ranked (`position_tier` in page_3_5/deep) *and* old (`age_tier` 181+ days), then prioritize the flagged pages with the most impressions. That is a completely reasonable, defensible rule, not a strawman.

It scores worse than the base rate. Individually, every signal I checked (position, age, CTR, engagement, freshness) has a weak correlation with the label (all under 0.17 in magnitude), so no single condition, or simple AND of two conditions, isolates the declining pages. The signal is real (54% of the catalog really is declining) but it's spread thinly across many weak, tangled signals rather than concentrated in one or two obvious ones, which is exactly the case where ML earns its place over an if-statement. For reference, this repo's own trained model (`outputs/model_report.md`) reaches 0.740 precision@50 versus 0.240 for its rule-based baseline on this same dataset, a gap this notebook's own numbers below help explain.

In [6]:
rule_flag = df['position_tier'].isin(['page_3_5', 'deep']) & df['age_tier'].isin(['181-365', '365+'])
flagged = df[rule_flag]
print("rule flags:", len(flagged), "rows")

top50_rule = flagged.sort_values('impressions_90d', ascending=False).head(50)
rule_precision_at_50 = top50_rule['is_declining_label'].mean()
base_rate = df['is_declining_label'].mean()
print(f"rule precision@50: {rule_precision_at_50:.3f}")
print(f"base rate (guessing):  {base_rate:.3f}")
print(f"-> the rule is essentially indistinguishable from guessing.\n")

feature_cols = ['avg_position','content_age_days','days_since_last_update','word_count',
                'ctr','engagement_rate','scroll_rate','ai_traffic_pct','impressions_90d']
corrs = df[feature_cols + ['is_declining_label']].corr()['is_declining_label'].drop('is_declining_label')
print("individual feature correlations with is_declining_label:")
print(corrs.reindex(corrs.abs().sort_values(ascending=False).index).round(3))

rule flags: 5752 rows
rule precision@50: 0.540
base rate (guessing):  0.542
-> the rule is essentially indistinguishable from guessing.

individual feature correlations with is_declining_label:
content_age_days         -0.164
word_count                0.090
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
impressions_90d          -0.018
engagement_rate          -0.013
scroll_rate              -0.003
ai_traffic_pct            0.002
Name: is_declining_label, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.